In [ ]:
# === QuakeMLab sample configuration (auto-added) ===
# By default the workflow runs on the bundled 5-event sample in data/sample/.
# To use your own archive, set QUAKEMLAB_SAMPLE or edit SFILE_PATH/EVENTS_DIR.
import os
SAMPLE = os.environ.get("QUAKEMLAB_SAMPLE") or next(
    (p for p in ("../data/sample", "data/sample") if os.path.isdir(p)), "data/sample")
SFILE_PATH = os.path.join(SAMPLE, "bulletin.out")      # Nordic (SEISAN) S-file
EXCEL_PATH = os.path.join(SAMPLE, "event_list.xlsx")   # optional AFAD list (absent for sample)
EVENTS_DIR = os.path.join(SAMPLE, "events")            # one folder per event, *.mseed (UTC)
DEMO_MODE  = True                                       # M4: run a single lightweight model
def models_to_run(full_list):
    return ["PhaseNetLight"] if DEMO_MODE else full_list
print("SAMPLE =", SAMPLE, "| exists:", os.path.isdir(SAMPLE))


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import json
import numpy as np
import pandas as pd
from obspy import read, UTCDateTime, Stream
import warnings
import seisbench.models as sbm
from tqdm import tqdm
import time

warnings.filterwarnings("ignore")

In [ ]:
mseed_dir = EVENTS_DIR
mseed_root = mseed_dir   # MSEED dosyalarının olduğu klasör
target_sr = 100.0                   # Hz, PhaseNet için
json_file = "sfile_parsed.json"

output_dir = "Preds"
# Picking modelleri
picking_models_bitti = []
picking_models = ["PhaseNet",
    "PhaseNetLight",
    "EQTransformer",
    "LFEDetect",
    "OBSTransformer",
    "Skynet",
    "BasicPhaseAE",
    "GPD",
]
picking_models_demo = [
    "PhaseNetLight"
]

mseed_files = []
for root, dirs, files in os.walk(mseed_dir):
    for f in files:
        if f.lower().endswith(".mseed"):
            mseed_files.append(os.path.join(root, f))

print(f"📂 Toplam {len(mseed_files)} MSEED dosyası bulundu.")

In [ ]:
import os
import obspy
from obspy.core.stream import Stream # Stream sınıfını doğrudan import etmeliyiz
import pandas as pd
from tqdm import tqdm
import warnings
import seisbench.models as sbm 
from obspy import read
import json 
import time 

# --- Varsayılan Değişkenlerin Tanımlanması (Sizin Ortamınıza göre ayarlayın) ---
# output_dir = "Preds" 
# target_sr = 100.0 # Varsayılıyor
# mseed_files = [...] # Varsayılıyor

picking_models_demo = ["PhaseNetLight"]

os.makedirs(output_dir, exist_ok=True) 

for m_name in models_to_run(picking_models):
    try:
        ModelClass = getattr(sbm, m_name)
    except AttributeError:
        warnings.warn(f"⚠️ SeisBench model {m_name} bulunamadı. Atlanıyor.")
        continue
        
    print(f"\n🔹 {m_name} pretrained modelleri işleniyor...") # Sadece başlangıç mesajı
    model_list = ModelClass.list_pretrained()
    for load_model in model_list:
        output_csv = os.path.join(output_dir, f"{m_name}_{load_model}.csv")
        #output_csv_single = os.path.join(output_dir, f"{m_name}_{load_model}_single.csv")
        
        # ----------------------------------------------------
        # 🎯 OPTİMİZASYON: Çıktı Dosyası Kontrolü (Tekrar Hesaplamayı Engelle)
        # ----------------------------------------------------
        if os.path.exists(output_csv):
            # print kaldırıldı, sadece uyarı verelim
            # print(f"  ✅ {m_name} ({load_model}) için CSV dosyası zaten var. Tahmin atlanıyor.")
            continue
        
        # print(f"  ⏳ {m_name} ({load_model}) yükleniyor ve tahmin yapılıyor...") # Kaldırıldı

        start_time = time.time()
        
        try:
            model = ModelClass.from_pretrained(load_model)
            model.eval()
        except Exception as e:
            warnings.warn(f"❌ {m_name} ({load_model}) yüklenirken hata: {e}")
            continue

        all_predictions = []

        # -------------------------------
        # 5️⃣ Dosya Bazlı Tahmin Döngüsü
        # -------------------------------
        # OPTİMİZASYON: tqdm tamamen kaldırıldı veya disable=True yapıldı.
        # Bu, terminal I/O'yu önemli ölçüde azaltır.
        for fpath in tqdm(mseed_files, desc=f"{m_name} {load_model}", disable=True): 
            fname = os.path.basename(fpath)
            
            try:
                st = read(fpath)
            except Exception as e:
                warnings.warn(f"⚠️ {fname} okunamadı: {e}")
                continue
            
            # --- SAĞLAM PREPROCESSING BAŞLANGICI ---
            filtered_st = Stream() # Obspy Stream sınıfı kullanıldı
            
            try:
                st.merge(method=1, fill_value=0) 
            except:
                pass 
                
            for tr in st:
                if abs(tr.stats.sampling_rate - target_sr) > 1e-2:
                    try:
                        tr.resample(target_sr)
                    except:
                        continue 
                        
                min_samples = int(target_sr * 10) 
                if tr.stats.npts < min_samples:
                    continue 

                tr.detrend("demean")
                tr.filter("bandpass", freqmin=1.0, freqmax=45.0)
                
                filtered_st.append(tr)
            
            st = filtered_st 
            if len(st) == 0:
                 continue
            # --- SAĞLAM PREPROCESSING SONU ---

            # tahmini
            try:
                picks = model.classify(st)
                
                if hasattr(picks, "picks") and len(picks.picks) > 0:
                    for pick in picks.picks:
                        event_id = fname.split('_')[0]
                        # ---------------------------------
                        # BAŞLANGIÇ: Güvenli Trace ID Çözümlemesi
                        # ---------------------------------
                        network = None
                        station = None
                        channel = None
                        component = None

                        try:
                            # 1. YÖNTEM (En Güvenli): Doğrudan trace'i bul
                            # pick.trace_id, 'st' (filtrelenmiş stream) içindeki tr.id'lerden biri olmalı
                            matching_trace = st.select(id=pick.trace_id)[0]
                            stats = matching_trace.stats
                            network = stats.network
                            station = stats.station
                            channel = stats.channel
                            component = stats.channel[-1].upper() # HHZ -> Z
                        
                        except (IndexError, Exception):
                            # 2. YÖNTEM (Manuel Ayrıştırma): Eşleşme bulunamazsa ID'yi parse et
                            try:
                                # 'KO.GULT..HHZ' -> ['KO', 'GULT', 'HHZ']
                                # 'KO.GULT..' -> ['KO', 'GULT']
                                # Boş string'leri ('') listeden kaldıran ayrıştırma:
                                id_parts = [part for part in pick.trace_id.split(".") if part] 

                                network = id_parts[0] if len(id_parts) > 0 else st[0].stats.network
                                station = id_parts[1] if len(id_parts) > 1 else st[0].stats.station
                                
                                # Eğer 3 parça varsa (N, S, C), sonuncusu kanaldır
                                if len(id_parts) >= 3:
                                    channel = id_parts[2]
                                else:
                                    # 3. YÖNTEM (Tahmin): Kanal bilgisi ID'de yoksa, 
                                    # P için 'Z' kanalı, S için 'N' veya 'E' kanalı ara
                                    if station: # En azından istasyon biliniyorsa
                                        comp_to_find = 'Z' if pick.phase.upper() == 'P' else '[NE]'
                                        fallback_trace = st.select(station=station, component=comp_to_find)
                                        if fallback_trace:
                                            channel = fallback_trace[0].stats.channel
                                        
                                if channel:
                                    component = channel[-1].upper()

                            except Exception:
                                # Son çare: Stream'deki ilk trace'in bilgilerini kullan
                                if len(st) > 0:
                                    network = st[0].stats.network
                                    station = st[0].stats.station
                                    channel = st[0].stats.channel
                                    component = st[0].stats.channel[-1].upper()
                        
                        # ---------------------------------
                        # SON: Güvenli Trace ID Çözümlemesi
                        # ---------------------------------
                        
                        confidence = float(getattr(pick, "confidence", getattr(pick, "peak_value", 1.0)))

                        all_predictions.append({
                            "source_file": fname, "model": f"{m_name}", "pretrained": f"{load_model}",
                            "event_id": event_id, "network": network, "station": station,
                            "channel": channel, "component": component, "phase": pick.phase,
                            "pick_time": str(pick.peak_time), "confidence": round(confidence,2)
                        })
            except Exception as e:
                # warnings.warn(f"⚠️ {fname} için {m_name} tahmin hatası: {e}") # Çok sık oluyorsa bu uyarıyı da kaldırabilirsiniz.
                pass 

        # -------------------------------
        # 6️⃣ CSV olarak kaydet
        # -------------------------------
        
        end_time = time.time()
        duration = round(end_time - start_time, 2)
        
        if all_predictions:
            df_pred = pd.DataFrame(all_predictions)
            df_pred['duration_sec'] = duration # Süreyi ekle
            df_pred['duration_sec'] = pd.to_numeric(df_pred['duration_sec'], errors='coerce')
            df_pred.to_csv(output_csv, index=False)
            
            # --- Tekil Pick Seçimi ---
            #df_picks = df_pred.copy()
            #group_keys = ['network', 'station', 'phase'] 
            #idx = df_picks.groupby(group_keys)['confidence'].idxmax()
            #df_unique = df_picks.loc[idx].reset_index(drop=True)
            #df_unique.to_csv(output_csv_single, index=False)
            
            # Başarılı tamamlama mesajını tek bir satırda ver
            #print(f"  ✅ TAMAMLANDI: {m_name} ({load_model}). Benzersiz Pick Sayısı: {len(df_unique)}. Toplam Süre: {duration:.2f} saniye.")

        else:
            # Boş sonuç mesajını tek bir satırda ver
            print(f"  ❌ BOŞ SONUÇ: {m_name} ({load_model}). Toplam Süre: {duration:.2f} saniye.")

## Npy dosyalarından tahmin

In [ ]:
import os
import pandas as pd
import numpy as np
import obspy
from obspy import UTCDateTime
import seisbench.models as sbm
from tqdm import tqdm
import time
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. AYARLAR
# ==========================================
# Hazırlanmış veri klasörü ve CSV dosyası
NPY_ROOT_DIR = "seisbench_ready"
INDEX_CSV = os.path.join(NPY_ROOT_DIR, "seisbench_ready.csv")
OUTPUT_DIR = "Preds_npy"

# Modeller
picking_models_to_run = ["PhaseNet",
    "PhaseNetLight",
    "EQTransformer",
    "LFEDetect",
    "OBSTransformer",
    "Skynet",
    "BasicPhaseAE",
    "GPD"] 

print("="*80)
print("🚀 SEISBENCH TAHMİN SÜRECİ (NPY MODU - FAST)")
print("="*80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. VERİ İNDEKSİNİ YÜKLE
# ==========================================
if not os.path.exists(INDEX_CSV):
    raise FileNotFoundError(f"❌ İndeks dosyası bulunamadı: {INDEX_CSV}. Lütfen önce SeisBenchPreparer çalıştırın.")

print("📂 Veri indeksi yükleniyor...")
df_index = pd.read_csv(INDEX_CSV)

# Event bazlı gruplama yapalım (Her event bir Stream olacak)
# Bir event içinde birden fazla istasyon/kanal olabilir.
grouped_events = df_index.groupby('event_id')

print(f"✅ {len(df_index)} trace, {len(grouped_events)} event olarak gruplandı.")

# ==========================================
# 3. YARDIMCI FONKSİYON: NPY -> OBSPY TRACE
# ==========================================
def load_stream_from_group(group_df, root_dir):
    """
    Pandas grubu (bir event'e ait satırlar) alır, 
    .npy dosyalarını okur ve Obspy Stream oluşturur.
    """
    st = obspy.Stream()
    
    for _, row in group_df.iterrows():
        try:
            # Dosya yolunu oluştur (CSV'deki yol relative olabilir)
            # full_path CSV'de varsa onu kullan, yoksa waveform_file'ı root ile birleştir
            npy_path = row.get('full_path')
            if not npy_path or not os.path.exists(str(npy_path)):
                npy_path = os.path.join(root_dir, row['waveform_file'])
            
            if not os.path.exists(npy_path):
                continue

            # NPY Oku
            data = np.load(npy_path)
            
            # Trace Oluştur (Metadata'yı CSV'den alıyoruz)
            header = {
                'network': row['network'],
                'station': row['station'],
                'location': str(row['location']) if pd.notna(row['location']) else '',
                'channel': row['channel'],
                'sampling_rate': row['sampling_rate'],
                'starttime': UTCDateTime(row['start_time'])
            }
            
            tr = obspy.Trace(data=data, header=header)
            st.append(tr)
            
        except Exception:
            continue
            
    return st

# ==========================================
# 4. TAHMİN DÖNGÜSÜ
# ==========================================

for m_name in models_to_run(picking_models_to_run):
    try:
        ModelClass = getattr(sbm, m_name)
    except AttributeError:
        continue
        
    print(f"\n🔹 {m_name} işleniyor...")
    model_list = ModelClass.list_pretrained()
    
    for load_model in model_list[:1]: # Demo için ilk model
        
        output_csv = os.path.join(OUTPUT_DIR, f"{m_name}_{load_model}.csv")
        
        if os.path.exists(output_csv):
             print(f"  ✅ {output_csv} zaten var. Atlanıyor.")
             continue
        
        print(f"  ⏳ Model yükleniyor: {load_model}")
        start_time = time.time()
        
        try:
            model = ModelClass.from_pretrained(load_model)
            model.eval()
        except Exception as e:
            print(f"❌ Model yükleme hatası: {e}")
            continue

        all_predictions = []

        # --- EVENT DÖNGÜSÜ ---
        # TQDM ile eventleri dönüyoruz
        for event_id, group_df in tqdm(grouped_events, desc=f"{m_name} {load_model}"):
            
            # 1. Stream'i oluştur (NPY'den oku)
            # Preprocessing (Filtre vb.) yapmıyoruz çünkü NPY'ler zaten işlenmiş!
            st = load_stream_from_group(group_df, NPY_ROOT_DIR)
            
            if len(st) == 0:
                continue
            
            # 2. Tahmin
            try:
                # SeisBench modelleri Stream objesi ile çalışır
                picks = model.classify(st)
                
                if hasattr(picks, "picks") and len(picks.picks) > 0:
                    for pick in picks.picks:
                        
                        # --- Trace ID Çözümleme (v7 - Indeksli Yöntem) ---
                        network, station, channel, component, location = None, None, None, None, None
                        
                        if hasattr(pick, 'trace_index_in_stream') and pick.trace_index_in_stream < len(st):
                            stats = st[pick.trace_index_in_stream].stats
                            network, station, location, channel = stats.network, stats.station, stats.location, stats.channel
                            
                            # Component Standardizasyonu
                            if len(stats.channel) > 0:
                                component = 'H' + stats.channel[-1].upper()
                            else:
                                component = stats.channel.upper()
                        
                        confidence = float(getattr(pick, "confidence", getattr(pick, "peak_value", 1.0)))
                        
                        # Süre hesabı (Stream'den al)
                        dur = st[0].stats.endtime - st[0].stats.starttime

                        all_predictions.append({
                            "source_file": f"Event_{event_id}", # Kaynak artık dosya değil Event ID
                            "model": f"{m_name}", 
                            "pretrained": f"{load_model}",
                            "event_id": str(event_id), 
                            "network": network,
                            "station": station,
                            "location": location,
                            "channel": channel, 
                            "component": component,
                            "phase": pick.phase,
                            "pick_time": str(pick.peak_time), 
                            "confidence": round(confidence,2),
                            "mseed_duration_sec": dur
                        })

            except Exception:
                pass

        # 3. Kaydet
        run_time = round(time.time() - start_time, 2)
        
        if all_predictions:
            df_pred = pd.DataFrame(all_predictions)
            df_pred['model_run_duration_sec'] = run_time
            # Excel format düzeltmesi
            df_pred['mseed_duration_sec'] = pd.to_numeric(df_pred['mseed_duration_sec'], errors='coerce').round(2).astype(str).replace('nan', '')
            
            df_pred.to_csv(output_csv, index=False)
            print(f"  ✅ Kaydedildi: {len(df_pred)} picks. Süre: {run_time}s")
        else:
            print(f"  ❌ Boş sonuç. Süre: {run_time}s")

print("\n🏁 İŞLEM TAMAMLANDI.")